# Project starter: build and defend a clinical ML model

Pick one track, reuse the pipeline pattern from Lab 1, Lab 2 and HW1, and produce a 6-slide
defence deck. Every numbered section below maps to a line of the marking rubric.

| Track | Dataset | Task |
|---|---|---|
| T1 | heart-statlog | classification |
| T2 | Indian Liver Patient | classification |
| T3 | scikit-learn diabetes progression | regression |
| T4 | SciPy ECG | unsupervised, use the ECG code-along notebook instead of this one |
| T5 | breast cancer diagnostic | classification, at most 5 features |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.datasets import load_breast_cancer, load_diabetes
from sklearn.model_selection import (train_test_split, StratifiedKFold, KFold,
                                     cross_validate, cross_val_predict)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import (RandomForestClassifier, HistGradientBoostingClassifier,
                              RandomForestRegressor, HistGradientBoostingRegressor)
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.inspection import permutation_importance
from sklearn.calibration import calibration_curve
from sklearn.metrics import (confusion_matrix, roc_auc_score, average_precision_score,
                             brier_score_loss, mean_absolute_error, root_mean_squared_error,
                             ConfusionMatrixDisplay)


def data_dir():
    """The data/ folder of this repository, wherever the notebook is opened from."""
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "data").is_dir():
            return p / "data"
    raise FileNotFoundError("Could not find the data folder. Run this notebook from inside the cloned repository.")


DATA = data_dir()
RANDOM_STATE = 42

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

## 1. Problem framing

Before any code, write three sentences in this cell:

- Who uses the output, and at what moment in the patient pathway?
- What decision does it change? If the answer is none, change the framing now, not on Day 3.
- What does a false negative cost? What does a false positive cost?

In [ ]:
def load_track(track):
    """Return X, y, task_type for one project track. All five use real patient data."""
    track = track.upper()

    if track == "T1":                                   # heart disease, Cleveland Clinic
        df = pd.read_csv(DATA / "heart_cleveland.csv")
        return df.drop(columns=["target"]), df["target"], "classification"

    if track == "T2":                                   # diabetes onset, Pima study
        df = pd.read_csv(DATA / "pima_diabetes.csv")
        return df.drop(columns=["diabetes"]), df["diabetes"], "classification"

    if track == "T3":                                   # one-year disease progression
        df = pd.read_csv(DATA / "diabetes_progression.csv")
        return df.drop(columns=["progression"]), df["progression"], "regression"

    if track == "T5":                                   # breast tumour diagnosis
        df = pd.read_csv(DATA / "breast_cancer_wdbc.csv")
        y = (df["diagnosis"] == "M").astype(int)        # malignant is the positive class
        return df.drop(columns=["diagnosis"]), y, "classification"

    raise ValueError("Use T1, T2, T3 or T5 here. T4 is the ECG track and starts from ecg_codealong.ipynb.")


TRACK = "T1"                                            # TODO: change to your track
X, y, task_type = load_track(TRACK)

# The splitter has to match the task. StratifiedKFold on a continuous target does not
# raise an error, it silently produces nonsense folds, so this line matters.
cv = (StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
      if task_type == "classification"
      else KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE))
scoring = (["roc_auc", "average_precision"] if task_type == "classification"
           else ["neg_mean_absolute_error", "neg_root_mean_squared_error"])

print(TRACK, task_type, X.shape)
print(pd.Series(y).value_counts().to_dict() if task_type == "classification"
      else pd.Series(y).describe().round(1).to_dict())
display(X.head())

## 2. Data handling

Cleaning goes inside the pipeline. Anything you compute on the full dataset before splitting
(a median, a scaler, a feature selection step) has leaked, and the examiner will ask.

In [ ]:
num_cols = X.select_dtypes(include='number').columns.tolist()
cat_cols = [c for c in X.columns if c not in num_cols]

preprocess = ColumnTransformer([
    ('num', Pipeline([('impute', SimpleImputer(strategy='median')),
                      ('scale', StandardScaler())]), num_cols),
    ('cat', Pipeline([('impute', SimpleImputer(strategy='most_frequent')),
                      ('onehot', OneHotEncoder(handle_unknown='ignore'))]), cat_cols),
])

# TODO: report size, missing values per column, class balance or target distribution
# TODO: say in one line what each impossible or missing value means clinically

## 3. Baseline and two real models

A trivial baseline is not optional. It is the number that tells the room whether 0.82 is good.

In [ ]:
if task_type == 'classification':
    models = {
        'dummy': Pipeline([('prep', preprocess), ('model', DummyClassifier(strategy='most_frequent'))]),
        'logreg': Pipeline([('prep', preprocess), ('model', LogisticRegression(max_iter=2000, class_weight='balanced'))]),
        'rf': Pipeline([('prep', preprocess), ('model', RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, class_weight='balanced'))]),
        'hgb': Pipeline([('prep', preprocess), ('model', HistGradientBoostingClassifier(random_state=RANDOM_STATE))]),
    }
else:
    models = {
        'dummy': Pipeline([('prep', preprocess), ('model', DummyRegressor(strategy='mean'))]),
        'ridge': Pipeline([('prep', preprocess), ('model', Ridge(alpha=1.0))]),
        'rf': Pipeline([('prep', preprocess), ('model', RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE))]),
        'hgb': Pipeline([('prep', preprocess), ('model', HistGradientBoostingRegressor(random_state=RANDOM_STATE))]),
    }

# TODO: keep the baseline plus at least two real models, drop the rest
# TODO: cross_validate each with cv and scoring above, report mean and std for every metric
# TODO: state which differences survive their own standard deviation

## 4. Operating point (classification) or error structure (regression)

Classification: choose the threshold on training data using cross-validated probabilities,
then apply it once to the test set. Justify the choice with the costs you wrote in section 1.

Regression: report MAE and RMSE in the units of the outcome, then look at the residuals.
Where the model is wrong matters more than how often.

In [ ]:
# TODO (classification): cross_val_predict probabilities on train, sweep thresholds,
#                       pick one from your cost model, confusion matrix on test
# TODO (regression): MAE and RMSE on test, residuals against the prediction, worst 5 cases

## 5. Calibration (classification tracks)

If anyone will read your output as a risk, or if you used costs to pick the threshold, the
probabilities have to mean something. Show the reliability curve and the Brier score.

In [ ]:
# TODO: calibration_curve(y_train, cv_proba, n_bins=10, strategy='quantile') plus the diagonal
# TODO: brier_score_loss for each candidate model
# TODO: one sentence on whether your chosen model is usable as a probability or only as a ranking

## 6. Interpretation and error analysis

In [ ]:
# TODO: permutation_importance on the test set for your chosen model
# TODO: for a linear model, coefficients as odds ratios; np.exp(coef) after scaling
# TODO: pull at least three individual errors and explain each one
# TODO: for each top feature, say whether it is a cause, a consequence, or a proxy

## 7. Limitations

Cover all four, briefly: who the population was, how small the sample is, what would shift in
a Malaysian hospital, and what you would need before anyone uses this on a patient.

## Final 6-slide deck

1. Problem and clinical user
2. Data: source, size, class balance, cleaning
3. Method and cross-validated results, with a baseline
4. Operating point or regression error, and why
5. What the model learned, plus error analysis
6. Limitations and next steps

6 minute talk, 4 minute questions. Every number on a slide must be reproducible from the
notebook you hand in.